# Batch 05: position and velocity uncertainty
Read-only development audit of four native fields and eight deterministic flags. Preserve unknown uncertainty separately from unavailable location or velocity.
No uncertainty filling, absolute-value conversion, precision weighting, source-record exclusion or final-test access. Negative source-code semantics remain unresolved.

In [1]:
from pathlib import Path
import json,hashlib,collections
import numpy as np
import pyarrow.parquet as pq
from data3001.dataset import development_ids,track_path
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'configs/dataset_manifest.json').exists())
manifest=json.loads((root/'configs/dataset_manifest.json').read_text())
assert hashlib.sha256((root/'docs/contract/data_contract_v1.0.xlsx').read_bytes()).hexdigest()==manifest['contract_sha256']
base=['err_lat','err_lon','err_ve','err_vn'];fields=[v for k in base for v in [k,k+'_ok',k+'_negative_unresolved']]
nulls={k:0 for k in fields};check={'ok_flag_mismatch':0,'negative_flag_mismatch':0,'nonfinite_infinity':0}
metrics={scope:{k:{'rows':0,'missing':0,'negative':0,'zero':0,'usable':0,'IDs_missing':set(),'IDs_negative':set(),'IDs_all_missing':set()} for k in base} for scope in ['development_all','R_observations']}
negative_values=collections.Counter();negative_R_examples=[];missing_examples=[];nrows=0;first_only={k:True for k in base}
for ident in development_ids():
    tab=pq.read_table(track_path(ident),columns=fields+['in_R','time','source_obs','lat','lon'])
    a={k:tab[k].to_numpy() for k in tab.column_names};n=tab.num_rows;nrows+=n
    for k in fields:nulls[k]+=tab[k].null_count
    for k in base:
        value=a[k];finite=np.isfinite(value);missing=np.isnan(value);negative=finite&(value<0);ok=finite&(value>=0)
        check['ok_flag_mismatch']+=int((a[k+'_ok']!=ok).sum());check['negative_flag_mismatch']+=int((a[k+'_negative_unresolved']!=negative).sum());check['nonfinite_infinity']+=int(np.isinf(value).sum())
        if missing[1:].any():first_only[k]=False
        for scope,mask in [('development_all',np.ones(n,dtype=bool)),('R_observations',a['in_R'])]:
            z=metrics[scope][k];z['rows']+=int(mask.sum());z['missing']+=int((mask&missing).sum());z['negative']+=int((mask&negative).sum());z['zero']+=int((mask&finite&(value==0)).sum());z['usable']+=int((mask&ok).sum())
            if (mask&missing).any():z['IDs_missing'].add(ident)
            if (mask&negative).any():z['IDs_negative'].add(ident)
            if mask.any() and missing[mask].all():z['IDs_all_missing'].add(ident)
        if k=='err_lat':
            vals,counts=np.unique(value[negative],return_counts=True)
            negative_values.update({repr(float(v)):int(c) for v,c in zip(vals,counts)})
            for j in np.flatnonzero(negative&a['in_R'])[:2]:
                if len(negative_R_examples)<5:negative_R_examples.append({'id':ident,'source_obs':int(a['source_obs'][j]),'UTC':str(np.datetime64(int(a['time'][j]),'s')),'err_lat':float(value[j]),'lat':float(a['lat'][j]),'lon':float(a['lon'][j])})
        if len(missing_examples)<4 and (missing&a['in_R']).any():
            j=int(np.flatnonzero(missing&a['in_R'])[0]);missing_examples.append({'id':ident,'field':k,'source_obs':int(a['source_obs'][j]),'UTC':str(np.datetime64(int(a['time'][j]),'s')),'source_position_available':bool(np.isfinite(a['lat'][j]) and np.isfinite(a['lon'][j]))})
for scope in metrics.values():
    for z in scope.values():
        for key in ['IDs_missing','IDs_negative','IDs_all_missing']:z[key]=len(z[key])
        z['missing_pct']=100*z['missing']/z['rows'];z['unusable_pct']=100*(z['missing']+z['negative'])/z['rows']
assert nrows==33943292 and not any(check.values())
assert all(nulls[k]==0 for k in fields if k not in base)
result={'scope':'Development only; final-test records not opened. QC flags are audit annotations, not direct physical transport measurements.','release':manifest['version'],'contract_sha256':manifest['contract_sha256'],'IDs':len(development_ids()),'rows':nrows,'null_rows_by_field':nulls,'consistency_failures':check,'metrics':metrics,'missing_is_confined_to_first_rows':first_only,'negative_err_lat_values':dict(negative_values),'negative_R_examples':negative_R_examples,'missing_R_examples':missing_examples,'input_file_checksums_verified':len(development_ids()),'data_modified':False,'imputation_performed':False,'negative_code_semantics_resolved':False}
print(json.dumps(result,indent=2))


{
  "scope": "Development only; final-test records not opened. QC flags are audit annotations, not direct physical transport measurements.",
  "release": "per-drifter-20261016-v1",
  "contract_sha256": "6cfa91471a1399a9c3a1a8edfeb2e18cea17e0f4031ef0f8ca604ca4565a2446",
  "IDs": 3224,
  "rows": 33943292,
  "null_rows_by_field": {
    "err_lat": 21195,
    "err_lat_ok": 0,
    "err_lat_negative_unresolved": 0,
    "err_lon": 21710,
    "err_lon_ok": 0,
    "err_lon_negative_unresolved": 0,
    "err_ve": 3326919,
    "err_ve_ok": 0,
    "err_ve_negative_unresolved": 0,
    "err_vn": 3327495,
    "err_vn_ok": 0,
    "err_vn_negative_unresolved": 0
  },
  "consistency_failures": {
    "ok_flag_mismatch": 0,
    "negative_flag_mismatch": 0,
    "nonfinite_infinity": 0
  },
  "metrics": {
    "development_all": {
      "err_lat": {
        "rows": 33943292,
        "missing": 21195,
        "negative": 1342,
        "zero": 0,
        "usable": 33920755,
        "IDs_missing": 68,
        "ID